# 📈 IDX Bandarmology — End-to-End Notebook

Notebook ini menjalankan seluruh alur: **scrape → bersihkan data → pipeline → analisis → modeling**,
untuk menguji hipotesis:

> *Apakah akumulasi "smart money" (bandar besar / broker asing) pada saham IDX memprediksi kenaikan harga berikutnya?*

**Sumber data:**
- `yfinance` — harga historis (OHLCV), gratis, tanpa token.
- `Stockbit` (exodus.stockbit.com) — broker summary, bandar detector, foreign/domestic flow. Butuh `STOCKBIT_TOKEN` di file `.env` (lihat `.env.example`).

**Cara pakai:** jalankan sel dari atas ke bawah. Edit `WATCHLIST` di sel berikutnya untuk ganti saham yang dipantau.
Jalankan notebook ini tiap hari bursa (atau jadwalkan) untuk menambah histori data — semakin banyak hari, semakin reliable hasil korelasi/modelingnya.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# repo root is the parent of this notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from idx_bandarmology import config, pipeline, storage, features, analysis, modeling, stockbit

print("Stockbit token configured:", stockbit.is_available())
print("Default watchlist:", config.WATCHLIST)

## 2. Watchlist

Edit daftar saham di bawah ini. Tidak perlu suffix `.JK` — itu otomatis ditambahkan untuk yfinance,
dan otomatis dibuang untuk Stockbit.

In [ ]:
WATCHLIST = ["BBCA", "BBRI", "BMRI", "BBNI", "TLKM", "ASII", "UNVR", "GOTO", "BREN", "ANTM"]

# Atau cari saham baru dulu sebelum menambah ke watchlist:
# from idx_bandarmology import stockbit
# stockbit.fetch_analysis("ADRO")["summary"]

## 3. Scrape data + jalankan pipeline

Satu fungsi ini melakukan semuanya:
1. Ambil harga historis dari yfinance untuk semua ticker di watchlist.
2. Ambil snapshot broker/bandar hari ini dari Stockbit (kalau token tersedia).
3. Simpan semuanya ke SQLite (`data/db/bandarmology.sqlite`) — ini jadi "pipeline" landing zone yang dipakai dashboard Streamlit juga.

Jalankan sel ini setiap hari bursa untuk menambah histori broker/bandar (Stockbit hanya kasih snapshot hari berjalan,
bukan range historis — jadi time series-nya terbentuk dari berapa kali kamu run pipeline ini).

In [ ]:
result = pipeline.run(WATCHLIST, price_period="2y")

print(f"Harga: {result['n_prices']} baris")
print(f"Broker/bandar: {result['n_broker']} baris")

### 3a. Cek hasil scrape mentah per ticker (opsional)

Berguna untuk verifikasi data Stockbit sebelum masuk pipeline — tiap section (`broker`, `foreignDomestic`,
`pricePerformance`) independen, jadi kalau satu gagal yang lain tetap tampil.

In [ ]:
for sym, r in result["broker_results"].items():
    print("=" * 70)
    print(sym, "-", r.get("summary"))
    if r.get("broker", {}).get("available"):
        print(" ", r["broker"]["conclusion"])
    if r.get("foreignDomestic", {}).get("available"):
        print(" ", r["foreignDomestic"]["conclusion"])

## 4. Olah data (pandas) — lihat tabel mentah dari pipeline

Dua tabel utama tersimpan di SQLite:
- `prices` — OHLCV harian per ticker (yfinance).
- `broker_flow` — snapshot broker/bandar per ticker per hari run (Stockbit).

In [ ]:
price_df = storage.read_prices(WATCHLIST)
broker_df = storage.read_broker_flow(WATCHLIST)

print("prices:", price_df.shape)
display(price_df.tail())

print("\nbroker_flow:", broker_df.shape)
display(broker_df.tail())

## 5. Feature engineering

`features.build_feature_table()` menggabungkan `prices` + `broker_flow` jadi satu tabel rapi,
plus menghitung **forward return** (return N hari ke depan dari hari sinyal muncul) — ini target
variable untuk uji hipotesis. Kita pakai forward return (bukan same-day) supaya tidak circular:
kita mau tahu apakah sinyal *hari ini* memprediksi harga *nanti*, bukan apakah pembelian besar
hari ini otomatis mendorong harga hari itu juga (yang memang mekanis selalu benar).

In [ ]:
feat = features.build_feature_table(WATCHLIST, horizons=(1, 3, 5, 10))
print(feat.shape)
feat.tail(10)[["ticker", "date", "close", "bandar_signal", "foreign_net_flow_pct",
               "fwd_return_1d", "fwd_return_5d", "fwd_return_10d"]]

## 6. Analisis deskriptif & korelasi

Pertanyaan inti: kalau hari ini sinyal bandar/asing menunjukkan **akumulasi**, apakah harga
benar-benar naik N hari kemudian (dibanding saat sinyalnya **distribusi** atau netral)?

In [ ]:
corr = analysis.correlation_table(feat)
display(corr)

In [ ]:
summary = analysis.summary_by_signal(feat, target_col="fwd_return_5d")
display(summary)

In [ ]:
fig = analysis.plot_signal_bucket_returns(feat, target_col="fwd_return_5d")
plt.show()

In [ ]:
fig = analysis.plot_signal_vs_forward_return(feat, horizon=5)
plt.show()

In [ ]:
# Korelasi per ticker — efeknya bisa berbeda-beda tiap saham
analysis.correlation_by_ticker(feat, target_col="fwd_return_5d")

In [ ]:
# Lihat harga + sinyal bandar untuk satu ticker
fig = analysis.plot_price_with_signal(feat, WATCHLIST[0])
plt.show()

## 7. Modeling — uji hipotesis statistik

Dua pendekatan:

1. **Regresi linier (OLS)** — apakah ada hubungan yang *signifikan secara statistik* (p < 0.05)
   antara sinyal smart money dan return ke depan, sambil mengontrol variabel lain?
2. **Klasifikasi (Logistic Regression / Random Forest)** — kalau kita ubah jadi pertanyaan biner
   ("naik atau tidak"), seberapa akurat sinyal smart money memprediksi itu?

In [ ]:
reg = modeling.linear_regression(feat, target_col="fwd_return_5d")
print(f"n = {reg.n_obs}, R-squared = {reg.r_squared:.4f}")
display(reg.coefficients)

In [ ]:
# Statsmodels summary lengkap (opsional, lebih detail)
print(reg.summary_text)

In [ ]:
clf_logit = modeling.classify_up_down(feat, target_col="fwd_return_5d", model_type="logistic")
print(f"Logistic Regression — n={clf_logit.n_obs}, accuracy={clf_logit.accuracy:.1%}, "
      f"precision={clf_logit.precision:.1%}, recall={clf_logit.recall:.1%}, "
      f"ROC-AUC={clf_logit.roc_auc}")
display(clf_logit.feature_importance)

In [ ]:
clf_rf = modeling.classify_up_down(feat, target_col="fwd_return_5d", model_type="random_forest")
print(f"Random Forest — n={clf_rf.n_obs}, accuracy={clf_rf.accuracy:.1%}, "
      f"precision={clf_rf.precision:.1%}, recall={clf_rf.recall:.1%}, "
      f"ROC-AUC={clf_rf.roc_auc}")
display(clf_rf.feature_importance)

## 8. Kesimpulan

In [ ]:
print(modeling.hypothesis_verdict(reg, clf_logit))

## 9. Dashboard

Untuk eksplorasi interaktif (filter ticker, lihat semua chart sekaligus, jalankan ulang pipeline
dari UI), buka dashboard Streamlit:

```bash
streamlit run dashboard/app.py
```

Dashboard membaca dari SQLite yang sama dengan notebook ini, jadi datanya selalu sinkron — tinggal
jalankan pipeline (dari notebook ini atau dari tombol di sidebar dashboard), lalu refresh.

> **Tips posting ke LinkedIn/GitHub:** screenshot tab "Modeling / Hipotesis" di dashboard, atau export
> salah satu chart di atas (`fig.savefig("chart.png", dpi=150)`) untuk dipakai sebagai cover image repo.